# Trabajo 1 - Econometría II 2026-1S
## Modelación ARIMA del Precio Internacional del Cobre

**Integrantes:** Santiago Vargas, Dilan Tovar, Maria Orjuela

**Profesores:** Milena Hoyos, Luis Luna, German Rodriguez

**Universidad Nacional de Colombia - Facultad de Ciencias Económicas**

---

### Descripción

Este notebook aplica la metodología Box-Jenkins para modelar el **precio internacional del cobre** (USD por tonelada métrica), con datos del **Fondo Monetario Internacional (FMI)** para el período marzo 2016 a marzo 2026 (121 observaciones mensuales).

### Motivación económica

El cobre, apodado **"Dr. Copper"**, es considerado un termómetro de la actividad económica global por su uso intensivo en construcción, manufactura, infraestructura eléctrica y la transición energética (vehículos eléctricos, energías renovables). Modelar su comportamiento tiene valor para la planeación industrial, las decisiones de inversión y el análisis macroeconómico.

### Estructura del notebook

0. **Configuración** — Importación de librerías y rutas relativas.
1. **Datos** — Lectura del CSV del FMI y preparación de la serie.
2. **Análisis exploratorio** — Visualización y transformación logarítmica.
3. **Identificación** — FAC, FACP y pruebas de raíz unitaria.
4. **Estimación** — Ajuste de modelos ARIMA candidatos.
5. **Validación** — Diagnóstico de residuos.
6. **Pronóstico** — Proyección a 10 meses.

### Fuente de los datos

Fondo Monetario Internacional (FMI), Primary Commodity Prices, serie *Global price of Copper* (PCOPPUSDM), obtenida vía FRED (Federal Reserve Bank of St. Louis). Unidades: dólares estadounidenses por tonelada métrica, no ajustado estacionalmente.

Archivo original: `data/raw/PCOPPUSDM.csv`

## Bloque 0: Configuración

Importación de paquetes y definición de rutas relativas usando `pathlib`. Esto garantiza que el código corra en cualquier computador sin modificar manualmente las rutas.

In [1]:
# %% Importación de paquetes ============================

# Trabajar con rutas relativas en python
from pathlib import Path

# Módulos de numpy, pandas, matplotlib y scipy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import jarque_bera, probplot

# Módulos de statsmodels
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller, kpss

# Configuración estética de matplotlib
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["figure.dpi"] = 100
plt.rcParams["savefig.dpi"] = 300
plt.rcParams["savefig.bbox"] = "tight"
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

print("Paquetes importados correctamente.")

Paquetes importados correctamente.


In [2]:
# %% Definición de rutas relativas =====================

# Detectar la raíz del proyecto:
# Si estamos ejecutando desde notebooks/, subir un nivel para llegar a la raíz.
# Si ya estamos en la raíz, usar el directorio actual.
notebook_path = Path.cwd()
if notebook_path.name == "notebooks":
    PROJECT_ROOT = notebook_path.parent
else:
    PROJECT_ROOT = notebook_path

# Definir carpetas estándar
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
FIGURES = PROJECT_ROOT / "outputs" / "figures"
TABLES = PROJECT_ROOT / "outputs" / "tables"

# Crear las carpetas si no existen (idempotente: no falla si ya existen)
for d in [DATA_PROCESSED, FIGURES, TABLES]:
    d.mkdir(parents=True, exist_ok=True)

# Verificar
print(f"Raíz del proyecto:  {PROJECT_ROOT}")
print(f"Datos crudos:       {DATA_RAW}")
print(f"Datos procesados:   {DATA_PROCESSED}")
print(f"Figuras:            {FIGURES}")
print(f"Tablas:             {TABLES}")

Raíz del proyecto:  c:\postergrupo_mds\poster
Datos crudos:       c:\postergrupo_mds\poster\data\raw
Datos procesados:   c:\postergrupo_mds\poster\data\processed
Figuras:            c:\postergrupo_mds\poster\outputs\figures
Tablas:             c:\postergrupo_mds\poster\outputs\tables


## Bloque 1: Carga y preparación de la serie

El archivo del FMI tiene una estructura simple con dos columnas:
- `observation_date`: fecha mensual (primer día de cada mes).
- `PCOPPUSDM`: precio del cobre en USD por tonelada métrica.

Vamos a:
1. Leer el CSV.
2. Renombrar columnas a nombres claros.
3. Construir el índice temporal mensual.
4. Guardar la serie como CSV procesado en `data/processed/cobre_precio.csv`.

In [4]:
# %% Lectura del CSV del FMI ============================

ruta_csv_raw = DATA_RAW / "PCOPPUSDM.csv"
print(f"Leyendo: {ruta_csv_raw}")

# Leer el CSV (estructura simple: fecha y precio)
df_raw = pd.read_csv(ruta_csv_raw)

print(f"Forma del DataFrame: {df_raw.shape}")
print(f"\nColumnas: {list(df_raw.columns)}")
print(f"\nPrimeras filas:")
print(df_raw.head())
print(f"\nÚltimas filas:")
print(df_raw.tail())

Leyendo: c:\postergrupo_mds\poster\data\raw\PCOPPUSDM.csv
Forma del DataFrame: (121, 2)

Columnas: ['observation_date', 'PCOPPUSDM']

Primeras filas:
  observation_date    PCOPPUSDM
0       2016-03-01  4953.797619
1       2016-04-01  4872.738095
2       2016-05-01  4694.537500
3       2016-06-01  4641.965909
4       2016-07-01  4864.904762

Últimas filas:
    observation_date     PCOPPUSDM
116       2025-11-01  10812.028000
117       2025-12-01  11790.964091
118       2026-01-01  12986.606818
119       2026-02-01  12951.345000
120       2026-03-01  12528.709545


In [5]:
# %% Construcción de la serie de tiempo =================

# Renombrar columnas a nombres claros
df = df_raw.rename(columns={
    "observation_date": "fecha",
    "PCOPPUSDM": "precio_cobre"
})

# Convertir la columna fecha a tipo datetime
df["fecha"] = pd.to_datetime(df["fecha"])

# Establecer fecha como índice y extraer la serie
cobre_serie = df.set_index("fecha")["precio_cobre"]

# Asegurar frecuencia mensual (primer día del mes)
cobre_serie = cobre_serie.asfreq("MS")

# Verificación
print(f"Observaciones: {len(cobre_serie)}")
print(f"Rango temporal: {cobre_serie.index.min().strftime('%Y-%m')} a {cobre_serie.index.max().strftime('%Y-%m')}")
print(f"\n¿Hay valores faltantes?: {cobre_serie.isna().sum()}")
print(f"\nPrimeras 5 observaciones:")
print(cobre_serie.head())
print(f"\nÚltimas 5 observaciones:")
print(cobre_serie.tail())

Observaciones: 121
Rango temporal: 2016-03 a 2026-03

¿Hay valores faltantes?: 0

Primeras 5 observaciones:
fecha
2016-03-01    4953.797619
2016-04-01    4872.738095
2016-05-01    4694.537500
2016-06-01    4641.965909
2016-07-01    4864.904762
Freq: MS, Name: precio_cobre, dtype: float64

Últimas 5 observaciones:
fecha
2025-11-01    10812.028000
2025-12-01    11790.964091
2026-01-01    12986.606818
2026-02-01    12951.345000
2026-03-01    12528.709545
Freq: MS, Name: precio_cobre, dtype: float64


In [6]:
# %% Estadísticas descriptivas =========================

print("Estadísticas descriptivas del precio del cobre (USD/tonelada):")
print(cobre_serie.describe().round(2))
print(f"\nValor mínimo: {cobre_serie.min():.2f} USD/ton ({cobre_serie.idxmin().strftime('%b %Y')})")
print(f"Valor máximo: {cobre_serie.max():.2f} USD/ton ({cobre_serie.idxmax().strftime('%b %Y')})")

Estadísticas descriptivas del precio del cobre (USD/tonelada):
count      121.00
mean      7728.41
std       1921.55
min       4641.97
25%       6031.21
50%       7772.24
75%       9324.82
max      12986.61
Name: precio_cobre, dtype: float64

Valor mínimo: 4641.97 USD/ton (Jun 2016)
Valor máximo: 12986.61 USD/ton (Jan 2026)


In [7]:
# %% Guardado del CSV procesado ========================

ruta_csv_proc = DATA_PROCESSED / "cobre_precio.csv"

# Convertir a DataFrame con dos columnas para guardar
cobre_df = cobre_serie.reset_index()
cobre_df.columns = ["fecha", "precio_cobre"]

# Guardar
cobre_df.to_csv(ruta_csv_proc, index=False)

print(f"CSV procesado guardado en: {ruta_csv_proc}")
print(f"Tamaño: {ruta_csv_proc.stat().st_size} bytes")
print(f"\nPrimeras 3 filas:")
print(cobre_df.head(3))

CSV procesado guardado en: c:\postergrupo_mds\poster\data\processed\cobre_precio.csv
Tamaño: 3381 bytes

Primeras 3 filas:
       fecha  precio_cobre
0 2016-03-01   4953.797619
1 2016-04-01   4872.738095
2 2016-05-01   4694.537500


## Bloque 2: Análisis exploratorio

Visualización inicial de la serie y evaluación de la necesidad de transformación logarítmica. Para precios de commodities es común que la variabilidad sea proporcional al nivel (heterocedasticidad multiplicativa), en cuyo caso el logaritmo natural estabiliza la varianza y linealiza el crecimiento.

Analizamos:
- La trayectoria del precio en niveles.
- La trayectoria en logaritmos.
- Los retornos logarítmicos mensuales (variabilidad).
- La presencia o ausencia de estacionalidad.

## Bloque 2: Análisis exploratorio

Visualización inicial de la serie y evaluación de la necesidad de transformación logarítmica. Para precios de commodities es común que la variabilidad sea proporcional al nivel (heterocedasticidad multiplicativa), en cuyo caso el logaritmo natural estabiliza la varianza y linealiza el crecimiento.

Analizamos:
- La trayectoria del precio en niveles.
- La trayectoria en logaritmos.
- Los retornos logarítmicos mensuales (variabilidad).
- La presencia o ausencia de estacionalidad.